<a href="https://colab.research.google.com/github/Paulokw/Text-Message-Spam-Detection-with-LSTM/blob/main/Text_message_spam_detection_with_LSTM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
#

In [ ]:
df=pd.read_csv(r'/content/SPAM text message 20170820 - Data.csv')
df.head()

,Category,Message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [ ]:
df['Category']=df['Category'].map({'ham':0,'spam':1})
df.head()

,Category,Message
0,0,"Go until jurong point, crazy.. Available only ..."
1,0,Ok lar... Joking wif u oni...
2,1,Free entry in 2 a wkly comp to win FA Cup fina...
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives aro..."


In [ ]:
import re
import nltk
from nltk.stem import WordNetLemmatizer

nltk.download('wordnet')

lem = WordNetLemmatizer()
corpus=[]
for review in df['Message']:
  review=re.sub("[^a-zA-Z0-9]", " ", review)
  review=review.lower()
  words=review.split()
  sent=[lem.lemmatize(word) for word in words]
  corpus.append(' '.join(sent))


[nltk_data] Downloading package wordnet to /root/nltk_data...


In [ ]:
corpus[0]

'go until jurong point crazy available only in bugis n great world la e buffet cine there got amore wat'

In [ ]:
# One-hot encoding
import tensorflow
import keras
from tensorflow.keras.preprocessing.text import one_hot

# Tokenization and padding
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Deep learning layers
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Dense, Dropout, LSTM

In [ ]:
tok=Tokenizer()
tok.fit_on_texts(corpus)
idx=tok.index_word
len(idx)

8206

In [ ]:
vocab_size=8210
one_hot_repr=[one_hot(word,vocab_size)for word in corpus]
sent_len=max([len(i) for i in one_hot_repr])
embedded_docs=pad_sequences(one_hot_repr,padding='pre',maxlen=sent_len)
embedded_docs[0]

array([   0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
          0,    0,    0,    0,    0,    0,    0,   

In [ ]:
model = Sequential()

model.add(Embedding(vocab_size, 50, input_length=sent_len))
model.add(LSTM(100))
model.add(Dropout(0.3))
model.add(Dense(1, activation='sigmoid'))
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
print(model.summary())

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_3 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

None


In [ ]:
X=np.array(embedded_docs)
y=np.array(df['Category'])

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)

In [ ]:
model.fit(X_train,y_train,validation_data=(X_test,y_test),epochs=10,batch_size=64)

Epoch 1/10
59/59 ━━━━━━━━━━━━━━━━━━━━ 8s 23ms/step - accuracy: 0.9006 - loss: 0.2864 - val_accuracy: 0.9777 - val_loss: 0.1012
Epoch 2/10
59/59 ━━━━━━━━━━━━━━━━━━━━ 5s 16ms/step - accuracy: 0.9815 - loss: 0.0695 - val_accuracy: 0.9842 - val_loss: 0.0573
Epoch 3/10
59/59 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.9920 - loss: 0.0326 - val_accuracy: 0.9902 - val_loss: 0.0455
Epoch 4/10
59/59 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.9960 - loss: 0.0177 - val_accuracy: 0.9886 - val_loss: 0.0467
Epoch 5/10
59/59 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.9976 - loss: 0.0118 - val_accuracy: 0.9908 - val_loss: 0.0447
Epoch 6/10
59/59 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.9989 - loss: 0.0064 - val_accuracy: 0.9880 - val_loss: 0.0446
Epoch 7/10
59/59 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 0.9992 - loss: 0.0044 - val_accuracy: 0.9897 - val_loss: 0.0466
Epoch 8/10
59/59 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - accuracy: 0.9992 - loss: 0.0037 - val_accuracy: 0.9897 - v

In [ ]:
y_pred=model.predict(X_test)

58/58 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step


In [ ]:
from sklearn.metrics import confusion_matrix
print(confusion_matrix(y_test,y_pred.round()))

[[1589    4]
 [  20  226]]
